# Week 52, Day 1: Greedy algorithms

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w052d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def greedy_change(amount, coins):
    used = []
    for c in sorted(coins, reverse=True):
        while amount >= c:
            amount -= c
            used.append(c)
    return used

print(greedy_change(68, [1, 2, 5, 10, 20, 50]))
print(greedy_change(6, [1, 3, 4]))

In [ ]:
def fractional_knapsack(items, capacity):
    """items: (name, value, weight). Returns the best total value and what was taken."""
    total, taken = 0.0, []
    for name, value, weight in sorted(items, key=lambda it: it[1] / it[2], reverse=True):
        if capacity <= 0:
            break
        amount = min(weight, capacity)
        total += value * amount / weight
        taken.append((name, amount))
        capacity -= amount
    return round(total, 2), taken

goods = [("rice", 60, 10), ("beans", 100, 20), ("saffron", 120, 3), ("salt", 10, 5)]
print(fractional_knapsack(goods, 25))

In [ ]:
from itertools import permutations

def total_finish(durations):
    t, total = 0, 0
    for d in durations:
        t += d
        total += t
    return total

jobs = [6, 2, 8, 3]
print("shortest first:", total_finish(sorted(jobs)))
print("best of every order:", min(total_finish(p) for p in permutations(jobs)))
print("longest first:", total_finish(sorted(jobs, reverse=True)))

In [ ]:
from functools import lru_cache

def fewest_coins(amount, coins):
    @lru_cache(maxsize=None)
    def best(a):
        if a == 0:
            return 0
        return min((best(a - c) + 1 for c in coins if c <= a), default=float("inf"))
    return best(amount)

def greedy_count(amount, coins):
    count = 0
    for c in sorted(coins, reverse=True):
        count += amount // c
        amount %= c
    return count

for coins in ([1, 2, 5, 10, 20, 50], [1, 3, 4], [1, 5, 12]):
    bad = [a for a in range(1, 60) if greedy_count(a, coins) != fewest_coins(a, coins)]
    print(coins, "greedy fails for", bad[:5] or "no amount below 60")

In [ ]:
def max_lateness(jobs):
    """jobs: (name, duration, deadline). Earliest deadline first."""
    t, worst, order = 0, 0, []
    for name, duration, deadline in sorted(jobs, key=lambda j: j[2]):
        t += duration
        worst = max(worst, t - deadline)
        order.append(name)
    return order, worst

tasks = [("report", 3, 6), ("slides", 2, 8), ("email", 1, 9), ("budget", 4, 9), ("call", 3, 14)]
print(max_lateness(tasks))

In [ ]:
def fewest_jumps(nums):
    jumps, current_end, farthest = 0, 0, 0
    for i in range(len(nums) - 1):
        farthest = max(farthest, i + nums[i])
        if i == current_end:              # the range reachable with `jumps` jumps is used up
            if farthest <= i:
                return None               # stuck
            jumps += 1
            current_end = farthest
    return jumps

print(fewest_jumps([2, 3, 1, 1, 4]), fewest_jumps([3, 2, 1, 0, 4]), fewest_jumps([0]))

> **Key idea.** A greedy algorithm takes the locally best choice at each step and never revisits it. It's optimal when the greedy choice can always start an optimal solution and what remains is a smaller problem of the same kind; most greedy algorithms are "sort by the right key, then take what fits". Prove a greedy rule with an exchange argument (swap any optimal solution towards the greedy one without making it worse), and test it against brute force on small inputs. Greedy works for fractional knapsack (value per kilo), total finishing time (shortest first) and maximum lateness (earliest deadline first); it fails for odd coin systems and 0/1 knapsack, where dynamic programming is needed.

### Exercise: Change, greedily

Write `greedy_change(amount, coins)`: a dict from coin to how many of it are used, taking the largest coin that fits each time (`ValueError` if the amount can't be made exactly). Then `is_canonical(coins, limit)`: whether greedy gives the fewest coins for every amount from 1 to `limit`, compared with a dynamic-programming table of fewest coins (`best[a] = 1 + min(best[a - c])`), built bottom-up.

In [ ]:
def greedy_change(amount, coins):
    ___

def is_canonical(coins, limit):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_coins():\n    \"coins\"\n    assert greedy_change(68, [1, 2, 5, 10, 20, 50]) == {50: 1, 10: 1, 5: 1, 2: 1, 1: 1}\n    assert greedy_change(0, [1, 5]) == {} and greedy_change(15, [5, 10]) == {10: 1, 5: 1}\n    try:\n        greedy_change(3, [2, 5])\n        assert False\n    except ValueError:\n        pass\n    assert is_canonical([1, 2, 5, 10, 20, 50, 100, 200], 500)\n    assert not is_canonical([1, 3, 4], 10) and not is_canonical([1, 5, 12], 20) and is_canonical([1, 5, 12], 14)")

### Exercise: Fractional knapsack

Write `fractional_knapsack(items, capacity)` for items `(name, value, weight)`: the best total value (as a float, rounded to 2 places) when any fraction of an item may be taken, and a list of `(name, fraction_taken)` with fractions between 0 and 1, best value per weight first. Then `zero_one_greedy(items, capacity)`: the value the same order gives when items must be taken **whole** (skipping any that don't fit), to see it fall short.

In [ ]:
def fractional_knapsack(items, capacity):
    ___

def zero_one_greedy(items, capacity):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools\ndef test_knapsack():\n    \"knapsack\"\n    goods = [(\"rice\", 60, 10), (\"beans\", 100, 20), (\"saffron\", 120, 3), (\"salt\", 10, 5)]\n    total, taken = fractional_knapsack(goods, 25)\n    assert total == 240.0 and taken == [(\"saffron\", 1.0), (\"rice\", 1.0), (\"beans\", 0.6)]\n    assert fractional_knapsack(goods, 0) == (0.0, [])\n    items = [(\"a\", 6, 6), (\"b\", 5, 5), (\"c\", 5, 5)]\n    assert zero_one_greedy(items, 10) == 6\n    best = max(sum(v for _, v, w in combo) for r in range(4) for combo in itertools.combinations(items, r) if sum(w for _, _, w in combo) <= 10)\n    assert best == 10 and fractional_knapsack(items, 10)[0] == 10.0")

### Exercise: Shortest job first

Customers at a counter each need a number of minutes. Write `best_order(jobs)` for jobs `(name, minutes)`: the order of names minimising the total of everyone's finishing times (shortest first; ties by name), and `total_waiting(jobs, order)`: the total of finishing times for a given order. Then `weighted_order(jobs)` for jobs `(name, minutes, weight)`, minimising the total of weight × finishing time: order by weight ÷ minutes, largest first (ties by name).

In [ ]:
def best_order(jobs):
    ___

def total_waiting(jobs, order):
    ___

def weighted_order(jobs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef wtotal(jobs, order):\n    info = {n: (m, w) for n, m, w in jobs}\n    t = total = 0\n    for n in order:\n        t += info[n][0]\n        total += info[n][1] * t\n    return total\ndef test_order():\n    \"orders\"\n    jobs = [(\"ada\", 6), (\"bo\", 2), (\"cy\", 8), (\"di\", 3)]\n    assert best_order(jobs) == [\"bo\", \"di\", \"ada\", \"cy\"] and total_waiting(jobs, best_order(jobs)) == 2 + 5 + 11 + 19\n    rng = random.Random(1)\n    for _ in range(50):\n        js = [(f\"j{i}\", rng.randint(1, 9), rng.randint(1, 5)) for i in range(5)]\n        best = min(wtotal(js, p) for p in itertools.permutations([n for n, _, _ in js]))\n        assert wtotal(js, weighted_order(js)) == best\n        plain = [(n, m) for n, m, _ in js]\n        assert total_waiting(plain, best_order(plain)) == min(total_waiting(plain, p) for p in itertools.permutations([n for n, _ in plain]))")

### Exercise: Earliest deadline first

Jobs are `(name, duration, deadline)`, done one after another from time 0. Write `least_worst_lateness(jobs)`: `(order, worst)`, where `worst` is the largest lateness (finish − deadline, at least 0) with earliest deadline first (ties by name). Then `most_on_time(jobs)`: the largest number of jobs that can all finish by their deadlines, using **Moore and Hodgson's** greedy: take jobs by deadline, and whenever the running time passes the current job's deadline, drop the **longest** job taken so far (a max-heap of durations).

In [ ]:
import heapq

def least_worst_lateness(jobs):
    ___

def most_on_time(jobs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef test_lateness():\n    \"lateness\"\n    tasks = [(\"report\", 3, 6), (\"slides\", 2, 8), (\"email\", 1, 9), (\"budget\", 4, 9), (\"call\", 3, 14)]\n    assert least_worst_lateness(tasks) == ([\"report\", \"slides\", \"budget\", \"email\", \"call\"], 1)\n    assert least_worst_lateness([]) == ([], 0)\n    rng = random.Random(2)\n    for _ in range(80):\n        js = [(f\"j{i}\", rng.randint(1, 5), rng.randint(1, 12)) for i in range(rng.randint(1, 6))]\n        best = 0\n        for r in range(len(js) + 1):\n            for combo in itertools.combinations(js, r):\n                t, ok = 0, True\n                for _, d, dl in sorted(combo, key=lambda j: j[2]):\n                    t += d\n                    ok = ok and t <= dl\n                if ok:\n                    best = max(best, r)\n        assert most_on_time(js) == best\n        worst = min(max([0] + [sum(j[1] for j in p[:k + 1]) - p[k][2] for k in range(len(p))]) for p in itertools.permutations(js))\n        assert least_worst_lateness(js)[1] == worst")

### Exercise: Jumping along

Write `fewest_jumps(nums)`: the fewest jumps from index 0 to the last index, where from index i you may jump forward up to `nums[i]` places, or `None` if the end can't be reached, in one O(n) greedy sweep. Then `can_reach_end(nums)`, tracking only the farthest reachable index, and `jump_path(nums)`: the indices landed on along one fewest-jumps route (starting with 0).

In [ ]:
def fewest_jumps(nums):
    ___

def can_reach_end(nums):
    ___

def jump_path(nums):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nfrom collections import deque\ndef bfs(nums):\n    n = len(nums)\n    d = {0: 0}\n    q = deque([0])\n    while q:\n        i = q.popleft()\n        for j in range(i + 1, min(n, i + nums[i] + 1)):\n            if j not in d:\n                d[j] = d[i] + 1\n                q.append(j)\n    return d.get(n - 1)\ndef test_jumps():\n    \"jumps\"\n    assert fewest_jumps([2, 3, 1, 1, 4]) == 2 and fewest_jumps([3, 2, 1, 0, 4]) is None and fewest_jumps([0]) == 0\n    assert can_reach_end([2, 3, 1, 1, 4]) and not can_reach_end([3, 2, 1, 0, 4])\n    assert jump_path([2, 3, 1, 1, 4]) == [0, 1, 4] and jump_path([3, 2, 1, 0, 4]) is None\n    rng = random.Random(3)\n    for _ in range(300):\n        nums = [rng.randint(0, 3) for _ in range(rng.randint(1, 10))]\n        assert fewest_jumps(nums) == bfs(nums) and can_reach_end(nums) == (bfs(nums) is not None)\n        p = jump_path(nums)\n        if p is not None:\n            assert p[0] == 0 and p[-1] == len(nums) - 1 and len(p) - 1 == bfs(nums)\n            assert all(0 < b - a <= nums[a] for a, b in zip(p, p[1:]))")

### Bug hunt: The wrong key

This fractional knapsack takes items in order of **value**, not value per kilo, so heavy, valuable items crowd out light, precious ones: this prints `232.0`. Sort by value ÷ weight so it prints `240.0`.

In [ ]:
goods = [("rice", 60, 10), ("beans", 100, 20), ("saffron", 120, 3), ("salt", 10, 5)]
capacity, total = 25, 0.0
for name, value, weight in sorted(goods, key=lambda it: it[1], reverse=True):
    amount = min(weight, capacity)
    total += value * amount / weight
    capacity -= amount
print(round(total, 2))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"value per kilo\"\n    expect_output(\"240.0\")")

### Bug hunt: Shortest isn't soonest

To minimise the worst lateness, this orders jobs by **duration** instead of deadline, so a job with an early deadline waits behind short ones: this prints `4`. Order by deadline so it prints `1`.

In [ ]:
tasks = [("report", 3, 6), ("slides", 2, 8), ("email", 1, 9), ("budget", 4, 9), ("call", 3, 14)]
t, worst = 0, 0
for name, duration, deadline in sorted(tasks, key=lambda j: j[1]):
    t += duration
    worst = max(worst, t - deadline)
print(worst)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"earliest deadline first\"\n    expect_output(\"1\")")

### Bug hunt: Trusting the greedy coins

This shop's coins are 1, 7 and 10, and it trusts greedy change, which gives 10 + 1 + 1 + 1 + 1 for 14 instead of 7 + 7. Replace the greedy count with the fewest-coins table (`best[a] = 1 + min(best[a - c])`) so it prints `2`.

In [ ]:
coins = [1, 7, 10]
amount = 14
count = 0
for c in sorted(coins, reverse=True):
    count += amount // c
    amount %= c
print(count)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"dynamic programming\"\n    expect_output(\"2\")")

### Puzzle: Fractional knapsack

Arrange the lines into the fractional knapsack. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the number of coins greedy uses for 6 with coins [1, 3, 4]
q = ___   # the fewest coins for 6 with coins [1, 3, 4]
r = ___   # the total of finishing times for jobs of 3, 1, 2 minutes done shortest first
s = ___   # the best value, fractional knapsack, capacity 4, items (value 10, weight 2) and (value 9, weight 3)
t = ___   # the fewest jumps for [1, 1, 1, 1]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 3 and q == 2 and r == 10, \"p, q and r\"\n    assert s == 16.0 and t == 3, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: jobs, deadlines and profits

Each job takes **one** day, has a deadline (the last day it may be done, from 1), and earns a profit if done in time. One job per day. Maximise the total profit. Greedy: take jobs from **most profitable** down, and give each the **latest free day** at or before its deadline (keeping earlier days for jobs with earlier deadlines); skip it if there's none.

Finding "the latest free day at or before d" by scanning is O(n) per job. With union-find it's nearly O(1): let `find(d)` return the latest free day ≤ d (0 means none); when day d is used, link d to d − 1, so later searches skip straight past it. Write `schedule(jobs)` for jobs `(name, deadline, profit)`, returning `(total_profit, plan)` where `plan` maps day numbers to job names; ties in profit are broken by name. The checks compare with brute force and run 100,000 jobs.

In [ ]:
def schedule(jobs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random, time\ndef brute(jobs):\n    best = 0\n    for r in range(len(jobs) + 1):\n        for combo in itertools.combinations(jobs, r):\n            ok = all(sum(1 for j in combo if j[1] <= d) <= d for d in range(1, 11))\n            if ok:\n                best = max(best, sum(j[2] for j in combo))\n    return best\ndef test_schedule():\n    \"schedules\"\n    jobs = [(\"a\", 2, 100), (\"b\", 1, 19), (\"c\", 2, 27), (\"d\", 1, 25), (\"e\", 3, 15)]\n    total, plan = schedule(jobs)\n    assert total == 142 and plan == {2: \"a\", 1: \"c\", 3: \"e\"}\n    assert schedule([]) == (0, {})\n    rng = random.Random(4)\n    for _ in range(80):\n        js = [(f\"j{i}\", rng.randint(1, 4), rng.randint(1, 20)) for i in range(rng.randint(0, 7))]\n        total, plan = schedule(js)\n        assert total == brute(js)\n        deadline = {n: d for n, d, _ in js}\n        assert all(day <= deadline[n] for day, n in plan.items())\ndef test_big():\n    \"fast\"\n    rng = random.Random(5)\n    js = [(f\"j{i}\", rng.randint(1, 50_000), rng.randint(1, 10**6)) for i in range(100_000)]\n    t0 = time.perf_counter()\n    total, plan = schedule(js)\n    assert len(plan) <= 50_000 and time.perf_counter() - t0 < 5")

## Warm-up

With ordinary coins, taking the biggest coin that fits gives the fewest coins: 50, 10, 5, 2, 1. With coins of 1, 3 and 4, it gives 4 + 1 + 1, three coins, but 3 + 3 is only two. The same greedy rule, right for one coin system and wrong for another.

## Take the best now

A greedy algorithm makes a sequence of choices, each the best available by some simple rule, and never undoes one. For it to give an optimal answer, the problem needs two properties:

- **Greedy-choice property**: some optimal solution begins with the greedy choice. So taking it never rules out the best answer.
- **Optimal substructure**: after the greedy choice, what's left is a smaller problem of the same kind, whose optimal solution completes an optimal solution to the whole.

When both hold, greedy is optimal, and it's usually just "sort by the right key, then take what fits". The work is in finding the key and proving it right.

**Fractional knapsack** is the cleanest example. You can carry a weight limit W; items have values and weights, and you may take any **fraction** of an item (flour, petrol, gold dust). Take items in order of **value per unit weight**, as much as fits, and split the last one. Since every bit of capacity goes to the most valuable stuff per kilo still available, nothing can do better.

## Proving it: the exchange argument

The standard proof that a greedy choice is safe is an **exchange argument**: take **any** optimal solution; if it differs from the greedy one, show you can **swap** a piece of it for the greedy choice without making it worse. Repeat, and the optimal solution turns into the greedy one, so greedy is optimal too.

For fractional knapsack: suppose an optimal load holds some of item B but not all the available amount of item A, where A has a higher value per kilo. Swap a little of B for the same weight of A. The weight is unchanged and the value goes **up** (or stays the same), so the swapped load is at least as good. Keep swapping until the load is the greedy one.

A neat example of the same idea: you have tasks with durations, done one after another, and want to minimise the **total waiting time** (the sum of each task's finishing time, say customers at a counter). Greedy: **shortest job first**. Exchange argument: if a longer job comes right before a shorter one, swapping them makes the shorter one finish earlier by the longer one's length and the longer one later by the shorter one's length; every other job is unaffected, so the total drops. So in an optimal order, no longer job precedes a shorter one.

Checking a greedy rule against every possibility on small inputs, as here, is the quick practical test; the exchange argument is the proof.

## When greed fails

Greed fails when an early choice that looks good blocks a better combination later. Learn the classic counterexamples; they're the fastest way to doubt a greedy idea:

- **Coins** {1, 3, 4}, amount 6: biggest first gives 4 + 1 + 1, but 3 + 3 is better. UK and euro coins happen to be **canonical** (greedy works for every amount), but not every coin system is.
- **0/1 knapsack** (each item whole or not at all): best value per kilo first can leave capacity unused. Capacity 10, items (value 6, weight 6) and two of (value 5, weight 5): greedy takes the 6 and stops at value 6; taking both 5s gives 10.
- **Longest path**: walking to the heaviest next edge each time can lead into a dead end.

Two habits help. Before trusting a greedy rule, try to **break** it with tiny inputs; and when a problem asks for a count or an optimum over choices that interact, suspect **dynamic programming** (the next two weeks), which considers every option systematically.

## Ordering jobs

Many greedy algorithms are "sort the jobs by the right key". The key depends on what's being optimised:

| goal | greedy order | why (exchange argument) |
|---|---|---|
| least total finishing time | shortest duration first | swapping a longer job before a shorter one lowers the total |
| least maximum lateness | **earliest deadline first** | swapping two jobs out of deadline order never makes the later-deadline job's lateness worse than the earlier one's was |
| most jobs finished by deadlines (unit durations) | by deadline, skipping jobs that no longer fit | |
| weighted total finishing time | largest weight ÷ duration first | the ratio rule, proved by the same swap |

Earliest deadline first, for minimising the worst lateness, is known as Jackson's rule (James Jackson, 1955).

## Reaching further

Some greedy algorithms don't sort at all; they sweep once, keeping the best "frontier" so far. **Jump game**: from position i you may jump up to `nums[i]` steps forward; what's the fewest jumps to reach the end? Track the farthest point reachable within the current number of jumps; when you pass the end of the current range, you must have jumped, and the next range ends at the farthest point seen. It's BFS by levels (Week 50) without a queue: O(n).

## Practice

## Quiz

## Daily challenge

## Before you go

Think of a decision you make greedily every day (the shortest queue at a shop, the first free seat). Can you invent a situation where it goes wrong? Tomorrow: interval problems, a family where greedy shines, with exchange arguments to prove it.